# ⚡ Fluxuri de lucru concurente cu agenți folosind modelele GitHub (Python)

## 📋 Tutorial avansat de procesare paralelă

Acest notebook demonstrează **modele de fluxuri de lucru concurente** folosind Microsoft Agent Framework. Vei învăța cum să construiești fluxuri de lucru de procesare paralelă performante, unde mai mulți agenți AI execută simultan, îmbunătățind dramatic debitul și permițând procese de afaceri multi-thread sofisticate.

## 🎯 Obiective de învățare

### 🚀 **Fundamentele procesării concurrente**
- **Execuția paralelă a agenților**: Rulează mai mulți agenți simultan pentru eficiență maximă
- **Orchestrarea fluxului de lucru**: Coordonează operațiuni concurente menținând consistența datelor
- **Optimizarea performanței**: Obține o creștere semnificativă a vitezei prin procesare paralelă
- **Gestionarea resurselor**: Utilizează eficient resursele modelelor AI în operațiuni concurente

### 🏗️ **Modele avansate de concurență**
- **Procesare fork-join**: Împarte munca între mai mulți agenți și combină rezultatele
- **Paralelism în pipeline**: Etape de execuție suprapuse pentru debit continuu
- **Echilibrare a încărcării**: Distribuie munca uniform pe resursele agenților disponibili
- **Puncte de sincronizare**: Coordonează agenții concurenți în etape critice ale fluxului

### 🏢 **Aplicații concurente pentru întreprinderi**
- **Procesare documente de volum mare**: Procesează simultan mai multe documente
- **Analiză de conținut în timp real**: Analiză concurentă a fluxurilor de date primite
- **Optimizarea procesării batch**: Maximizează debitul pentru operațiuni la scară largă
- **Analiză multimodală**: Procesare paralelă pentru tipuri diferite de conținut (text, imagini, date)

## ⚙️ Cerințe și configurare

### 📦 **Dependențe necesare**

Instalează Agent Framework cu capabilități de flux de lucru concurent:

```bash
pip install agent-framework-core -U
```

### 🔑 **Configurarea modelelor GitHub**

**Configurare mediu (.env file):**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

**Considerații pentru procesarea concurentă:**
- **Limite de rată**: Monitorizează limitele API GitHub Models pentru cereri concurente
- **Utilizarea resurselor**: Ia în calcul memoria și CPU folosite de mai mulți agenți concurenți
- **Gestionarea erorilor**: Implementează recuperare robustă la erori pentru operațiuni paralele

### 🏗️ **Arhitectura fluxului de lucru concurent**

```mermaid
graph TD
    A[Începerea fluxului de lucru] --> B[Execuție concurentă]
    B --> C[Grup de agenți 1]
    B --> D[Grup de agenți 2]
    B --> E[Grup de agenți 3]
    C --> F[Agregarea rezultatului]
    D --> F
    E --> F
    F --> G[Rezultatul final]
    
    H[API Modele GitHub] --> C
    H --> D
    H --> E
```

**Beneficii cheie:**
- **⚡ Performanță**: Creștere semnificativă a vitezei prin execuție paralelă
- **📈 Scalabilitate**: Gestionează creșterea volumului de lucru fără creștere proporțională a timpului
- **🔄 Eficiență**: Utilizare mai bună a resurselor computational disponibile
- **🎯 Debit**: Procesează mai multă muncă în același interval de timp

## 🎨 **Modele de design pentru fluxuri de lucru concurente**

### 🔍 **Pipeline de cercetare și analiză**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Flux de procesare a datelor**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Pipeline de creare de conținut**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Procesare în mai multe etape**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Beneficii de performanță pentru întreprinderi**

### ⚡ **Optimizarea debitului**
- **Execuție paralelă**: Mai mulți agenți lucrând simultan
- **Utilizarea resurselor**: Eficiență maximă a capacității modelelor AI disponibile
- **Reducerea timpului**: Scădere semnificativă a timpului total de procesare
- **Arhitectură scalabilă**: Adaugă cu ușurință mai mulți agenți concurenți după necesitate

### 🛡️ **Fiabilitate și reziliență**
- **Toleranță la erori**: Eșecurile agenților individuali nu opresc întregul flux de lucru
- **Izolare erori**: Problemele într-un fir concurent nu afectează celelalte
- **Degradare grațioasă**: Sistemul continuă să funcționeze chiar și cu capacitate redusă a agenților
- **Mecanisme de recuperare**: Retentative automate și gestionare a erorilor pentru operațiuni eșuate

### 📊 **Monitorizare și observabilitate**
- **Urmărire execuții concurente**: Monitorizează progresul tuturor operațiunilor paralele
- **Metrici de performanță**: Măsoară îmbunătățirea vitezei și eficienței
- **Analiza utilizării resurselor**: Optimizează alocarea agenților concurenți
- **Identificarea blocajelor**: Găsește și rezolvă constrângerile de performanță

Hai să construim fluxuri AI concurente de înaltă performanță! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = await provider.create_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = await provider.create_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)

In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
